# Classifieur CNN — entraînement (Colab)

CNN entraîné **de zéro** sur **Imagenette** (10 classes d'ImageNet, images réalistes), avec **Keras 3 + JAX**.

## 1. Récupérer le code du projet

In [ ]:
import os

REPO_URL = "https://github.com/tristan-reig/Projet_L3_REMAKE.git"
REPO_DIR = "/content/projet-ia-creative"

if not os.path.exists(REPO_DIR):
    !GIT_TERMINAL_PROMPT=0 git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("Répertoire de travail :", os.getcwd())

## 2. Backend JAX + dépendances


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
print("Keras :", keras.__version__, "| backend :", keras.backend.backend())
# Imagenette est chargé sans tensorflow_datasets : aucun conflit Protobuf.

## 3. Paramètres

In [ ]:
IMG_SIZE   = 160
BATCH_SIZE = 64
EPOCHS     = 30

MODEL_NAME = "model_classifier.keras"

## 4. (Optionnel) Monter Google Drive

Recommandé pour sauver les checkpoints au fil de l'entraînement (une déconnexion de session ne fait alors pas tout perdre).

In [ ]:
USE_DRIVE = True  # False -> reste en local Colab + download manuel à la fin

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/projet-ia-creative/models"
else:
    SAVE_DIR = "/content/projet-ia-creative/models"

os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, MODEL_NAME)
print("Le modèle sera sauvegardé ici :", MODEL_PATH)

## 5. Préparer le dataset Imagenette

Premier appel : téléchargement (~1.5 Go) puis mise en cache. Les splits train / validation d'Imagenette sont utilisés tels quels.

In [ ]:
from ml.classifier.data import build_dataset

train_ds, val_ds, class_names = build_dataset(batch_size=BATCH_SIZE)
num_classes = len(class_names)
print("Datasets prêts —", num_classes, "classes :", class_names)

## 6. Visualiser quelques images

Vérification rapide que les données et les labels sont cohérents.

In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(8):
    ax = plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[int(labels[i])], fontsize=9)
    plt.axis("off")
plt.tight_layout(); plt.show()

## 7. Construire le CNN

4 blocs convolutifs (32→64→128→256), GlobalAveragePooling, puis tête dense. La normalisation des pixels est intégrée au modèle (couche Rescaling), donc l'inférence web n'a rien à reproduire.

In [ ]:
from ml.classifier.model import build_cnn, compile_model

model = build_cnn(img_size=IMG_SIZE, num_classes=num_classes)
model = compile_model(model)
model.summary()

## 8. Entraîner

`ModelCheckpoint` garde le meilleur modèle (val_accuracy). `EarlyStopping` arrête si la validation stagne. `ReduceLROnPlateau` baisse le learning rate quand la perte plafonne.

In [ ]:
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 9. Courbes d'apprentissage

In [ ]:
import matplotlib.pyplot as plt

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (tr, val, title) in zip(
    axes,
    [("accuracy", "val_accuracy", "Exactitude"), ("loss", "val_loss", "Perte")],
):
    ax.plot(h[tr], label="Train", linewidth=2)
    ax.plot(h[val], label="Val", linewidth=2, linestyle="--")
    ax.set_title(title); ax.set_xlabel("Époque"); ax.legend()
    ax.grid(True, linestyle="--", alpha=0.6)
fig.tight_layout(); plt.show()

best = max(h["val_accuracy"])
print(f"Meilleure exactitude validation : {best:.1%}")

## 10. Matrice de confusion

Où le modèle se trompe-t-il ? Utile pour repérer les classes confondues.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true, y_pred = np.array(y_true), np.array(y_pred)
n = len(class_names)
cm = np.zeros((n, n), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(class_names, fontsize=8)
ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
for i in range(n):
    for j in range(n):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
fig.colorbar(im); plt.tight_layout(); plt.show()

## 11. Tester sur une image

Upload une image via le panneau Fichiers de Colab, puis indique son chemin.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from ml.classifier.model import IMG_SIZE

TEST_IMAGE = "/content/test.jpg"  # adapte le chemin

img = Image.open(TEST_IMAGE).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
x = np.expand_dims(np.asarray(img, dtype="float32"), 0)  # le Rescaling est dans le modèle
preds = model.predict(x, verbose=0)[0]
top = np.argsort(preds)[::-1][:3]

plt.imshow(img); plt.axis("off")
plt.title(" · ".join(f"{class_names[i]} {preds[i]:.0%}" for i in top))
plt.show()

## 12. Récupérer le modèle entraîné

Avec Drive, le `.keras` y est déjà. Sinon, télécharge-le et place-le dans `models/` de ton projet (suivi par Git LFS).

In [ ]:
if not USE_DRIVE:
    from google.colab import files
    files.download(MODEL_PATH)
else:
    print("Modèle disponible sur Drive :", MODEL_PATH)